# 📚 ETL com Python + Claude AI — Boletim Inteligente

> Pipeline **ETL completo** aplicado ao domínio da **Educação**.  
> Lê dados de alunos em CSV, usa o **Claude (Anthropic)** para gerar um boletim pedagógico personalizado por aluno e salva os resultados localmente.

---

## 🎯 Contexto

Você trabalha na coordenação pedagógica de uma escola e precisa enviar boletins individuais para cada aluno.  
Em vez de textos genéricos, você quer usar **IA Generativa** para criar um diagnóstico real — considerando as 12 disciplinas, faltas e ano — e uma mensagem motivacional única para cada estudante.

---

## 🔄 Fluxo ETL

```
📂 students.csv        🤖 Claude AI              💾 JSON + CSV
   (Extract)    ──►    (Transform)     ──►        (Load)

Notas e faltas      Boletim pedagógico        boletins_output
de cada aluno       personalizado por aluno   .json  e  .csv
```

| Etapa | O que acontece |
|-------|----------------|
| **Extract** | Lê `students.csv` com pandas |
| **Transform** | Claude gera diagnóstico, riscos, sugestões e mensagem por aluno |
| **Load** | Salva `boletins_output.json` e `boletins_output.csv` |

---

## ⚙️ Configuração

### 1. Instalar o SDK da Anthropic

In [ ]:
!pip install anthropic -q

### 2. Configurar a API Key

Para obter sua chave:
1. Acesse [console.anthropic.com](https://console.anthropic.com)
2. Vá em **API Keys** → **Create Key**
3. Cole abaixo substituindo `'SUA_CHAVE_AQUI'`

> 💡 **Dica de segurança:** No Colab, prefira usar `userdata` ou variáveis de ambiente para não expor sua chave no código.

In [ ]:
ANTHROPIC_API_KEY = 'SUA_CHAVE_AQUI'

# Alternativa mais segura no Colab:
# from google.colab import userdata
# ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')

# Ou via variável de ambiente:
# import os
# ANTHROPIC_API_KEY = os.environ.get('ANTHROPIC_API_KEY')

---

## 📥 E — Extract

Lê o arquivo `students.csv` e prepara os dados para a etapa de transformação.

**Estrutura do CSV — 12 disciplinas + informações do aluno:**

| Coluna | Descrição |
|--------|-----------|
| `Student` | Nome do aluno |
| `Year` | Ano escolar (1º, 2º ou 3º ano) |
| `Arts` | Nota em Artes (0–10) |
| `PhysicalEducation` | Nota em Educação Física (0–10) |
| `Philosophy` | Nota em Filosofia (0–10) |
| `Sociology` | Nota em Sociologia (0–10) |
| `English` | Nota em Inglês (0–10) |
| `Physics` | Nota em Física (0–10) |
| `Chemistry` | Nota em Química (0–10) |
| `Biology` | Nota em Biologia (0–10) |
| `Geography` | Nota em Geografia (0–10) |
| `History` | Nota em História (0–10) |
| `Math` | Nota em Matemática (0–10) |
| `Portuguese` | Nota em Língua Portuguesa (0–10) |
| `Absences` | Total de faltas no bimestre |

In [ ]:
import pandas as pd
import json

# Todas as 12 disciplinas (chave CSV → nome em português)
SUBJECTS = {
    'Arts':              'Artes',
    'PhysicalEducation': 'Educação Física',
    'Philosophy':        'Filosofia',
    'Sociology':         'Sociologia',
    'English':           'Inglês',
    'Physics':           'Física',
    'Chemistry':         'Química',
    'Biology':           'Biologia',
    'Geography':         'Geografia',
    'History':           'História',
    'Math':              'Matemática',
    'Portuguese':        'Língua Portuguesa'
}

# Leitura do CSV
df = pd.read_csv('students.csv')

print(f'✅ {len(df)} alunos carregados com sucesso!\n')
print(df[['Student', 'Year', 'Absences']].to_string(index=False))

In [ ]:
# Converte para lista de dicionários
# Calcula média geral e identifica a disciplina com menor e maior nota
students = df.to_dict(orient='records')

for s in students:
    s['AverageGrade']   = round(sum(s[sub] for sub in SUBJECTS) / len(SUBJECTS), 2)
    s['LowestSubject']  = min(SUBJECTS, key=lambda sub: s[sub])
    s['HighestSubject'] = max(SUBJECTS, key=lambda sub: s[sub])
    s['FailingSubjects'] = [sub for sub in SUBJECTS if s[sub] < 5.0]
    s['report'] = {}  # campo que receberá o boletim gerado pela IA

print('Prévia dos dados calculados:\n')
print(f'{"Aluno":<20} {"Ano":<10} {"Média":>6}  {"Faltas":>6}  {"Reprovando em"}')
print('-' * 72)
for s in students:
    failing = ', '.join(SUBJECTS[f] for f in s['FailingSubjects']) or 'nenhuma'
    print(f"{s['Student']:<20} {s['Year']:<10} {s['AverageGrade']:>6.1f}  {s['Absences']:>6}  {failing}")

---

## 🔄 T — Transform

O **Claude** age como um pedagogo especialista.  
Para cada aluno, ele recebe as 12 notas e as faltas e devolve um boletim estruturado com:

- **Diagnóstico** — avaliação geral do desempenho considerando todas as disciplinas
- **Riscos** — disciplinas ou situações que precisam de atenção urgente
- **Recomendações** — ações práticas e específicas de estudo
- **Mensagem motivacional** — frase personalizada e encorajadora para o aluno

> O modelo usado é o `claude-haiku-4-5-20251001` — rápido e ideal para geração de texto em escala.

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

SYSTEM_PROMPT = """Você é um pedagogo especialista em educação básica e ensino médio.
Analise os dados do aluno e retorne SOMENTE um objeto JSON válido, sem texto extra, sem markdown.
O JSON deve ter exatamente estas chaves:
  - diagnostico: string com avaliação geral do desempenho considerando todas as disciplinas (2-3 frases)
  - riscos: lista de strings com até 3 pontos de atenção prioritários (mencione disciplinas específicas)
  - recomendacoes: lista de strings com até 3 ações práticas de estudo (específicas por disciplina)
  - mensagem: string com frase motivacional curta e personalizada dirigida ao aluno (máx. 120 caracteres)
"""

def generate_boletim(student: dict) -> dict:
    """
    Envia os dados completos do aluno ao Claude e retorna o boletim como dicionário.
    """
    # Monta bloco de notas com todas as 12 disciplinas
    notas_str = '\n'.join(
        f"  {SUBJECTS[sub]}: {student[sub]:.1f}{'  ⚠️' if student[sub] < 5.0 else ''}"
        for sub in SUBJECTS
    )

    failing_names = ', '.join(SUBJECTS[f] for f in student['FailingSubjects']) or 'nenhuma'

    user_prompt = f"""Analise o boletim abaixo e gere o relatório pedagógico:

Aluno: {student['Student']}
Ano: {student['Year']}
Faltas no bimestre: {student['Absences']}
Média geral: {student['AverageGrade']:.1f}
Disciplinas com nota abaixo de 5,0: {failing_names}

Notas por disciplina:
{notas_str}
"""

    response = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=600,
        system=SYSTEM_PROMPT,
        messages=[
            {'role': 'user', 'content': user_prompt}
        ]
    )

    raw = response.content[0].text.strip()

    # Remove possíveis marcações markdown antes do parse
    raw = raw.replace('```json', '').replace('```', '').strip()

    return json.loads(raw)

In [ ]:
print('🤖 Gerando boletins com Claude...\n')
print('=' * 70)

for student in students:
    boletim = generate_boletim(student)
    student['report'] = boletim

    status = '🔴 Atenção' if student['FailingSubjects'] or student['Absences'] >= 10 else '🟢 Regular'

    print(f"\n👤 {student['Student']} — {student['Year']}  |  Média: {student['AverageGrade']:.1f}  |  Faltas: {student['Absences']}  |  {status}")
    print(f"\n   📋 Diagnóstico:")
    print(f"   {boletim.get('diagnostico', '')}")
    print(f"\n   ⚠️  Riscos:")
    for r in boletim.get('riscos', []):
        print(f"      • {r}")
    print(f"\n   📖 Recomendações:")
    for rec in boletim.get('recomendacoes', []):
        print(f"      • {rec}")
    print(f"\n   💬 Mensagem: {boletim.get('mensagem', '')}")
    print('\n' + '-' * 70)

print(f'\n✅ {len(students)} boletins gerados com sucesso!')

---

## 💾 L — Load

Salva os boletins gerados em dois formatos:

- `boletins_output.json` — estrutura completa, ideal para sistemas e APIs
- `boletins_output.csv` — formato tabular, ideal para planilhas e relatórios da coordenação

In [ ]:
# --- Salvar JSON completo ---
output_json = 'boletins_output.json'

with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(students, f, ensure_ascii=False, indent=2)

print(f'✅ JSON salvo em: {output_json}')

# --- Salvar CSV enriquecido ---
output_csv = 'boletins_output.csv'

rows = []
for s in students:
    r = s.get('report', {})
    row = {
        'Student':       s['Student'],
        'Year':          s['Year'],
        'AverageGrade':  s['AverageGrade'],
        'Absences':      s['Absences'],
    }
    # Adiciona todas as 12 disciplinas
    for sub, name in SUBJECTS.items():
        row[name] = s[sub]
    # Adiciona campos gerados pela IA
    row['Diagnostico']   = r.get('diagnostico', '')
    row['Riscos']        = ' | '.join(r.get('riscos', []))
    row['Recomendacoes'] = ' | '.join(r.get('recomendacoes', []))
    row['Mensagem']      = r.get('mensagem', '')
    rows.append(row)

df_output = pd.DataFrame(rows)
df_output.to_csv(output_csv, index=False, encoding='utf-8')

print(f'✅ CSV salvo em:  {output_csv}')
print()

# Exibe resumo final
cols_resumo = ['Student', 'Year', 'AverageGrade', 'Absences', 'Mensagem']
print(df_output[cols_resumo].to_string(index=False))

---

## 📊 Resumo do Pipeline

| Etapa | Entrada | Processamento | Saída |
|-------|---------|---------------|-------|
| **Extract** | `students.csv` | `pd.read_csv()` + cálculo de média, pior matéria e disciplinas abaixo de 5,0 | Lista de dicionários com campo `report: {}` |
| **Transform** | 12 notas + faltas por aluno | Claude gera JSON pedagógico personalizado | Campo `report` com diagnóstico, riscos, recomendações e mensagem |
| **Load** | Lista enriquecida | `json.dump()` + `df.to_csv()` | `boletins_output.json` + `boletins_output.csv` |

**Disciplinas contempladas:**
Artes · Educação Física · Filosofia · Sociologia · Inglês · Física · Química · Biologia · Geografia · História · Matemática · Língua Portuguesa

---

## 🚀 Próximos Passos

- 📧 **Enviar boletins por e-mail** — integre com SendGrid ou Gmail API para disparar automaticamente para os responsáveis
- 📄 **Gerar PDFs individuais** — use `fpdf2` ou `reportlab` para criar um arquivo por aluno com layout profissional
- 📊 **Visualizar desempenho por turma** — adicione gráficos de radar por disciplina com `plotly`
- 🗄️ **Persistir em banco de dados** — salve os boletins em SQLite com `sqlite3` para histórico bimestral
- 🔄 **Automatizar a cada bimestre** — agende com GitHub Actions ou `schedule`
- 🌐 **Criar uma API REST** — exponha os boletins via FastAPI para integração com sistemas escolares

---

*Desenvolvido com 🤖 [Claude (Anthropic)](https://www.anthropic.com) + 🐍 Python*